# 03 · Student Baseline — No KD

This notebook is reserved for the **student-only baseline**.

It must not read teacher probabilities or logits.

Protocol:
- fit on TRAIN-107 only
- select checkpoint on DEV-34 participant-level macro-F1
- participant 440 excluded
- threshold fixed by the approved student protocol
- TEST remains closed

The previous segment-teacher-dependent student runner is intentionally not executed here.


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists():
    subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## Preflight

This cell verifies the dataset protocol and refuses to run the obsolete student training path.


In [ ]:
DATA=Path('/content/drive/MyDrive/DAIC_WOZ')
import pandas as pd
def split(name):
    hits=list(DATA.rglob(name)); assert len(hits)==1,(name,hits)
    d=pd.read_csv(hits[0]); c={x.lower():x for x in d.columns}
    return d[c['participant_id']].astype(int)
tr=split('train_split_Depression_AVEC2017.csv')
dv=split('dev_split_Depression_AVEC2017.csv'); dv=dv[dv.ne(440)]
assert len(tr)==107 and len(dv)==34 and not set(tr)&set(dv)
assert not (ROOT/'scripts/teachers/train_segment_teachers.py').exists()
print('TRAIN=107 DEV=34; old segment teacher entry point absent; TEST untouched.')


## Student runner gate

The participant-correct student implementation will live under `scripts/students/participant_student/`. This notebook will run it only after that implementation is committed.


In [ ]:
RUNNER=ROOT/'scripts/students/participant_student/train_baseline.py'
assert RUNNER.exists(), (
 'Student migration is not committed yet. Do NOT run scripts/students/train_segment_students.py. '
 'Implement participant-target broadcasting and participant-level DEV selection first.'
)
subprocess.run([sys.executable,str(RUNNER)],cwd=ROOT,check=True)
